## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <vector>
#include <set>
#include <map>
#include <queue>
#include <string>
#include <algorithm>
#include <iostream>
#include <bitset>
#include <functional>
#include <numeric>
#include <cstdio>
#include <cstring>
#include <cstdlib>
#include <cassert>
#include <cmath>
#include <ctime>
#include <complex>
using namespace std;

typedef long long LL;
typedef complex<double> Complex;

#define fi first
#define se second
#define ins insert
#define pb push_back

inline int fastpo(int x, int n, int mod) {
    int res(1);
    while (n) {
        if (n & 1) {
            res = res * (LL)x % mod;
        }
        x = x * (LL)x % mod;
        n /= 2;
    }
    return res;
}

const int N = 1024;
const int LOG = 20;
const int mod = 1e9 + 7;
const int inf = 1e9 + 7;

vector<int> operations;   
int curPerm[N];           
int pos[N];               

int targetA, targetB;     
int lowbitLen;            
int permSize;             

void refresh() {
    for (int i = 0; i < permSize; ++i)
        pos[curPerm[i]] = i;
}

void doMagic() {
    operations.push_back(0);
    for (int i = 0; i < permSize; ++i) {
        if (curPerm[i] == targetA) curPerm[i] = targetB;
        else if (curPerm[i] == targetB) curPerm[i] = targetA;
    }
    refresh();
}

void doAdd(int x) {
    if (x == 0) return;
    operations.push_back(x);
    for (int i = 0; i < permSize; ++i)
        curPerm[i] = (curPerm[i] + x) % permSize;
    refresh();
}

void doXor(int x) {
    if (x == 0) return;
    operations.push_back(-x);
    for (int i = 0; i < permSize; ++i)
        curPerm[i] = (curPerm[i] ^ x);
    refresh();
}

void calc(int a, int b, int &pa, int &pb) {
    int delta = (b - a + permSize - lowbitLen + permSize) % permSize;
    pa = pb = 0;
    for (int step = permSize / 2; step >= 2 * lowbitLen; step /= 2) {
        if (delta >= step) {
            delta -= step;
            pb += step / 2;
        } else {
            pa += step / 2;
        }
    }
    pa += permSize / 2;
    pa += (a & (lowbitLen - 1));
    pb += (a & (lowbitLen - 1));
}

void doSwap(int c, int d) {
    if (c / lowbitLen % 2 == d / lowbitLen % 2) {
        int p;
        if (c / lowbitLen % 2 == 0)
            p = (c & (lowbitLen - 1)) + lowbitLen;
        else
            p = (c & (lowbitLen - 1));
        doSwap(c, p);
        doSwap(d, p);
        doSwap(c, p);
    } else {
        int pa, pb;
        calc(targetA, targetB, pa, pb);
        int pc, pd;
        calc(c, d, pc, pd);

        doAdd((pc - c + permSize) % permSize);
        doXor((pc ^ pa));
        doAdd((targetA - pa + permSize) % permSize);
        doMagic();
        doAdd((pa - targetA + permSize) % permSize);
        doXor((pc ^ pa));
        doAdd((c - pc + permSize) % permSize);
    }
}

struct Permutation {
    int data[N];
    int size;
    vector<int> vec;

    bool operator < (const Permutation &b) const {
        for (int i = 0; i < size; ++i)
            if (data[i] != b.data[i])
                return data[i] < b.data[i];
        return false;
    }

    void print() const {
        for (int i = 0; i < size; ++i)
            cout << data[i] << " \n"[i == size - 1];
    }

    Permutation inv() const {
        Permutation res;
        for (int i = 0; i < size; ++i)
            res.data[data[i]] = i;
        return res;
    }

    bool sorted() const {
        for (int i = 0; i + 1 < size; ++i)
            if (data[i] > data[i + 1])
                return false;
        return true;
    }

    bool calc() {
        bool f[100005] = {};
        for (int i = 0; i < size; ++i) f[i] = 1;
        for (int i = 0; i < size; ++i) if (!f[i]) return false;
        if (size == 1) return true;

        Permutation b, c;
        for (int i = 0; i < size / 2; ++i) {
            b.data[i] = data[i * 2] / 2;
            c.data[i] = data[i * 2 + 1] / 2;
        }
        b.size = size / 2;
        c.size = size / 2;

        if (!b.calc() || !c.calc()) return false;
        if (data[0] % 2) vec.push_back(size == 2 ? 1 : -1);

        int tb = 0;
        for (int i : b.vec) {
            if (i > 0) {
                vec.push_back(-1);
                vec.push_back(1);
            } else {
                vec.push_back(i * 2);
                tb ^= -i * 2;
            }
        }
        if (tb) vec.push_back(-tb);

        int tc = 0;
        for (int i : c.vec) {
            if (i > 0) {
                vec.push_back(1);
                vec.push_back(-1);
            } else {
                vec.push_back(i * 2);
                tc ^= -i * 2;
            }
        }
        if ((tc & (size / 2)) != (tb & (size / 2))) {
            assert(false);
            for (int i = 0; i < size / 4; ++i) {
                vec.push_back(-1);
                vec.push_back(1);
            }
        }
        if (tb >= (size / 2)) tb -= size / 2;
        if (tc >= (size / 2)) tc -= size / 2;
        if (tb != tc) return false;

        vector<int> tmp;
        for (int i : vec) {
            if (tmp.empty())
                tmp.push_back(i);
            else {
                if ((i < 0) && (tmp.back() < 0)) {
                    tmp.back() = - ((-tmp.back()) ^ (-i));
                    if (tmp.back() == 0) tmp.pop_back();
                } else {
                    tmp.push_back(i);
                }
            }
        }
        swap(tmp, vec);
        return true;
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    cin >> permSize >> targetA >> targetB;
    for (int i = 0; i < permSize; ++i)
        cin >> curPerm[i];

    refresh();
    lowbitLen = (targetA - targetB + permSize) % permSize;
    lowbitLen = lowbitLen & -lowbitLen;
    if (lowbitLen == 0) lowbitLen = permSize;

    if (lowbitLen > 1) {
        Permutation permObj;
        permObj.size = lowbitLen;
        for (int i = 0; i < permSize; ++i)
            permObj.data[i] = curPerm[i] & (lowbitLen - 1);

        if (!permObj.calc()) {
            cout << "-1\n";
            exit(0);
        }

        for (auto val : permObj.vec) {
            if (val > 0) doAdd(val);
            else doXor(-val);
        }
    }

    for (int i = 0; i < lowbitLen; ++i) {
        vector<int> vec;
        for (int j = i; j < permSize; j += lowbitLen)
            vec.push_back(curPerm[j]);

        int idx = 0, ok = 1;
        sort(vec.begin(), vec.end());
        for (int j = i; j < permSize; j += lowbitLen) {
            if (vec[idx] != j) {
                ok = 0;
                break;
            }
            ++idx;
        }
        if (!ok) {
            cout << "-1\n";
            exit(0);
        }

        for (int j = i; j < permSize; j += lowbitLen) {
            if (curPerm[j] != j)
                doSwap(j, curPerm[j]);
        }
    }

    for (int i = 0; i < permSize; ++i)
        assert(curPerm[i] == i);

    cout << operations.size() << '\n';
    for (int op : operations) {
        if (op == 0)
            cout << "0\n";
        else if (op < 0)
            cout << "1 " << -op << '\n';   
        else
            cout << "2 " << op << '\n';    
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here
import sys

def can_arrive(distance, limit, money, shop_list):
    # 初始体力已经可以直接到终点
    if limit >= distance:
        return True

    # 同一个位置只保留价格最低的补给站
    cheapest = {}
    for pos, cost in shop_list:
        if pos > distance:
            continue
        if pos not in cheapest or cost < cheapest[pos]:
            cheapest[pos] = cost

    points = sorted(cheapest.items())

    # 情况一：只补给一次
    for pos, cost in points:
        if pos <= limit and cost <= money and distance - pos <= limit:
            return True

    # 情况二：补给两次
    total = len(points)

    for first in range(total):
        pos_a, cost_a = points[first]

        # 第一个补给点已经超过最大可达距离，后面更不可能
        if pos_a > limit:
            break

        # 买不起第一站补给
        if cost_a > money:
            continue

        for second in range(first + 1, total):
            pos_b, cost_b = points[second]

            # 第一站到不了第二站，后面的第二站也到不了
            if pos_b - pos_a > limit:
                break

            # 第二站补给后到不了终点
            if distance - pos_b > limit:
                continue

            # 两次补给费用不超过预算
            if cost_a + cost_b <= money:
                return True

    return False


def main():
    data = sys.stdin.read().split()
    if not data:
        return

    cursor = 0
    result = []

    while cursor + 4 <= len(data):
        n = int(data[cursor])
        road_len = int(data[cursor + 1])
        max_run = int(data[cursor + 2])
        budget = int(data[cursor + 3])
        cursor += 4

        supplies = []
        for _ in range(n):
            position = int(data[cursor])
            price = int(data[cursor + 1])
            supplies.append((position, price))
            cursor += 2

        if can_arrive(road_len, max_run, budget, supplies):
            result.append("Yes")
        else:
            result.append("No")

    print("\n".join(result))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
## add your code here
#include<iostream>
using namespace std;

typedef unsigned long long ULL;

const int N = 1e5 + 10;
const int P = 131;

ULL powP[N];          
ULL prefixHash[N];    
ULL suffixHash[N];    
int radA[N * 2];     
int radB[N * 2];      
int strLen;           

    
// 在原字符串中插入特殊字符，用于 Manacher 算法
string transform(string& s) {
    string result = "(#";
    for (char ch : s) {
        result += ch;
        result += '#';
    }
    result += ')';
    return result;
}

// Manacher 算法，计算每个位置的回文半径
void manacher(string& s, int radius[]) {
    int right = 0, center = 0;
    int sz = s.size();
    for (int i = 1; i < sz - 1; ++i) {
        if (i < right)
            radius[i] = min(radius[2 * center - i], right - i);
        else
            radius[i] = 1;

        while (s[i - radius[i]] == s[i + radius[i]])
            ++radius[i];

        if (i + radius[i] > right) {
            right = i + radius[i];
            center = i;
        }
    }
}

// 初始化哈希：前缀哈希（a）和后缀哈希（b）
void initHash(string& a, string& b) {
    powP[0] = 1;
    for (int i = 1; i <= strLen; ++i) {
        powP[i] = powP[i - 1] * P;
    }

    for (int i = 1; i <= strLen; ++i) {
        prefixHash[i] = prefixHash[i - 1] * P + a[i - 1];
    }

    for (int i = strLen; i >= 1; --i) {
        suffixHash[i] = suffixHash[i + 1] * P + b[i - 1];
    }
}

// 二分查找最长公共前缀/后缀长度
// r1 : a 中右端点位置（1‑based）
// l2 : b 中左端点位置（1‑based，实际为 suffixHash 的索引）
int binarySearch(int r1, int l2) {
    int low = 0, high = min(r1, strLen - l2 + 1) + 1;
    while (low + 1 < high) {
        int mid = (low + high) / 2;
        ULL hashA = prefixHash[r1] - prefixHash[r1 - mid] * powP[mid];
        ULL hashB = suffixHash[l2] - suffixHash[l2 + mid] * powP[mid];
        if (hashA == hashB)
            low = mid;
        else
            high = mid;
    }
    return low;
}

int main() {
    string a, b;
    cin >> strLen >> a >> b;

    string transformedA = transform(a);
    string transformedB = transform(b);

    manacher(transformedA, radA);
    manacher(transformedB, radB);

    initHash(a, b);

    int ans = 0;
    int totalLen = strLen * 2 + 1;   

    for (int i = 1; i <= totalLen; ++i) {
        
        ans = max(ans, radA[i] - 1);
        int len = radA[i] - 1;
        int centerIdx = i / 2;          
        int r1 = centerIdx - len / 2 - 1;
        int l2 = centerIdx + len / 2 + 1;
        if (len % 2 == 0)
            ++r1;
        int commonLen = binarySearch(r1, l2 - 1);
        ans = max(ans, len + commonLen * 2);

       
        ans = max(ans, radB[i] - 1);
        len = radB[i] - 1;
        centerIdx = i / 2;
        r1 = centerIdx - len / 2 - 1;
        l2 = centerIdx + len / 2 + 1;
        if (len % 2 == 0)
            ++r1;
        commonLen = binarySearch(r1 + 1, l2);
        ans = max(ans, len + commonLen * 2);
    }

    printf("%d\n", ans);
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

const int MAX_X = 1e5 + 7;   // 优惠券编号 x 的最大值
int cnt[MAX_X];             // 记录每种优惠券当前的持有状态（0：未持有，1：已持有）
int lastTime[MAX_X];        // 记录每种优惠券上一次被操作（购买或使用）
set<int> conflictTimes;     // 记录所有 "?"（未知记录）

void init() {
    memset(cnt, 0, sizeof(cnt));
    memset(lastTime, 0, sizeof(lastTime));
    conflictTimes.clear();
}

void solve() {
    ios::sync_with_stdio(false);
    cin.tie(0);
    
    int m;
    string cmd;          
    while (cin >> m) {
        int firstError = 0; // 记录最早发生矛盾的
        init();
        for (int i = 1; i <= m; ++i) {
            cin >> cmd;
            if (cmd == "O" || cmd == "I") { // 如果是明确的 购买 (I) 或 使用 (O) 优惠券操作
                int x;
                cin >> x;
                cnt[x] += (cmd == "I" ? 1 : -1); // I (购买) 状态 +1，O (使用) 状态 -1
                // 检查是否发生逻辑冲突：
                // cnt[x] < 0 手里没有这张优惠券却尝试使用它
                // cnt[x] > 1 手里已经有这张券了，又重复购买
                if (cnt[x] < 0 || cnt[x] > 1) {
                    auto it = conflictTimes.lower_bound(lastTime[x]);
                    if (it == conflictTimes.end()) {
                        if (!firstError) firstError = i;
                    } else {
                        conflictTimes.erase(it);
                        cnt[x] = min(1, max(cnt[x], 0));
                    }
                }
                lastTime[x] = i;
            } else { // cmd == "?"
                conflictTimes.insert(i);
            }
        }
        cout << (firstError ? firstError : -1) << '\n';
    }
}

int main() {
    solve();
    return 0;
}

## E 任意点

In [ ]:
## add your code here
import sys

def get_root(father, index):
    # 查找当前节点所属集合的根节点
    if father[index] != index:
        father[index] = get_root(father, father[index])
    return father[index]


def merge(father, a, b):
    # 合并两个点所在的集合
    root_a = get_root(father, a)
    root_b = get_root(father, b)

    if root_a == root_b:
        return False

    father[root_a] = root_b
    return True


def main():
    data = sys.stdin.read().split()
    if not data:
        return

    total = int(data[0])
    node_list = []

    pos = 1
    for _ in range(total):
        x_coord = int(data[pos])
        y_coord = int(data[pos + 1])
        node_list.append((x_coord, y_coord))
        pos += 2

    # 初始化并查集
    father = [i for i in range(total)]
    group_count = total

    # 如果两个点横坐标相同或纵坐标相同，就认为它们可以连通
    for i in range(total):
        x1, y1 = node_list[i]

        for j in range(i + 1, total):
            x2, y2 = node_list[j]

            if x1 == x2 or y1 == y2:
                if merge(father, i, j):
                    group_count -= 1

    # 把所有连通块连接起来，至少需要 group_count - 1 个点
    print(group_count - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
# add your code here
#include <iostream>
#include <vector>
#include <string>
using namespace std;

struct Block {
    int len;
    vector<pair<int, string>> parts;
    int anchorOffset;
    string anchor;

    Block() {}

    Block(const string& s) {
        len = s.size();
        anchorOffset = 0;
        anchor = "";

        int i = 0;
        while (i < len) {
            if (s[i] == '?') { // 跳过 '?'
                i++;
            } else {
                int j = i;
                // 截取连续的普通字符
                while (j < len && s[j] != '?') {
                    j++;
                }

                string literal = s.substr(i, j - i);
                parts.push_back({i, literal}); // 记录这段纯字母子串和它的偏移量
                // 贪心策略：挑选最长的一段纯字母子串作为 "锚点" (Anchor)
                if (literal.size() > anchor.size()) {
                    anchor = literal;
                    anchorOffset = i;
                }

                i = j;
            }
        }
    }
};

// 检查目标字符串text从pos索引开始，能否匹配这个 Block
bool checkAt(const string& text, const Block& block, int pos) {
    if (pos < 0) return false;
    if (pos + block.len > (int)text.size()) return false;// 剩余长度不够
    
    // 遍历 Block 中的每一个字母片段，检查偏移位置是否一致
    for (auto& item : block.parts) {
        int offset = item.first;
        const string& literal = item.second;
        // compare 比较局部字符串
        if (text.compare(pos + offset, literal.size(), literal) != 0) {
            return false;
        }
    }
    // 所有纯字母片段都匹配上了
    return true;
}

int findBlock(const string& text, const Block& block, int left, int right) {
    // 在 text 的 [left, right) 范围内找 block 的最早匹配位置

    if (block.len == 0) {
        return left;
    }

    int maxStart = right - block.len; // Block 能放置的最晚起始位置
    if (left > maxStart) {
        return -1;
    }

    // 如果这个 block 全是 ?，例如 "???"
    if (block.parts.empty()) {
        return left;
    }
    
    // 寻找锚点的合法区间：计算锚点在 text 中的起始查找点和最晚边界
    int searchStart = left + block.anchorOffset;
    int lastAnchorStart = maxStart + block.anchorOffset;

    size_t pos = text.find(block.anchor, searchStart);

    while (pos != string::npos && (int)pos <= lastAnchorStart) {
        int blockStart = (int)pos - block.anchorOffset;

        if (checkAt(text, block, blockStart)) {
            return blockStart;
        }

        pos = text.find(block.anchor, pos + 1);
    }

    return -1;
}

// 根据 '*' 将 pattern 切割成多个子串
// 例如 "*aca?ctc*" 会被切分为 ["", "aca?ctc", ""]
vector<string> splitByStar(const string& pattern) {
    vector<string> blocks;
    string cur;

    for (char c : pattern) {
        if (c == '*') {
            blocks.push_back(cur);
            cur.clear();
        } else {
            cur.push_back(c);
        }
    }

    blocks.push_back(cur);
    return blocks;
}

bool isMatch(const string& filename, const string& pattern) {
    // 没有 * 的情况，长度必须相等
    if (pattern.find('*') == string::npos) {
        if (filename.size() != pattern.size()) {
            return false;
        }

        Block block(pattern);
        return checkAt(filename, block, 0);
    }
    
    // 有 '*' 的情况
    vector<string> rawBlocks = splitByStar(pattern);

    vector<Block> blocks;
    blocks.reserve(rawBlocks.size());

    for (const string& s : rawBlocks) {
        blocks.push_back(Block(s));
    }

    int left = 0;
    int right = filename.size();

    int startIndex = 0;
    int endIndex = (int)rawBlocks.size() - 1;

    // 如果 pattern 不是以 * 开头，第一段必须从文件名开头匹配
    if (!rawBlocks.front().empty()) {
        if (!checkAt(filename, blocks.front(), 0)) {
            return false;
        }

        left = blocks.front().len;
        startIndex = 1;
    } else {
        startIndex = 1;
    }

    // 如果 pattern 不是以 * 结尾，最后一段必须贴着文件名结尾匹配
    if (!rawBlocks.back().empty()) {
        const Block& lastBlock = blocks.back();
        int lastPos = (int)filename.size() - lastBlock.len;

        if (lastPos < left) {
            return false;
        }

        if (!checkAt(filename, lastBlock, lastPos)) {
            return false;
        }

        right = lastPos;
        endIndex = (int)rawBlocks.size() - 2;
    } else {
        endIndex = (int)rawBlocks.size() - 2;
    }

    // 中间的 block 只需要按顺序出现
    for (int i = startIndex; i <= endIndex; i++) {
        if (rawBlocks[i].empty()) {
            continue;
        }
        // 在剩余范围内寻找最早匹配当前 Block 的位置
        int pos = findBlock(filename, blocks[i], left, right);

        if (pos == -1) {
            return false;
        }

        left = pos + blocks[i].len;
    }

    return left <= right;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    int n;

    cin >> pattern;
    cin >> n;

    string filename;

    for (int i = 0; i < n; i++) {
        cin >> filename;

        if (isMatch(filename, pattern)) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [2]:
## add your code here
import sys

def Hanoi():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    priorities = input_data[1:7]
    
    # 建立优先级映射表，索引越小优先级越高
    prio_map = {move: i for i, move in enumerate(priorities)}
    
    # dp[i][u] = (steps, dest_peg)
    # 表示将 i 个盘子从柱子 u 移动完毕所需的步数，以及最终落在哪根柱子上
    # 柱子映射: 0 -> A, 1 -> B, 2 -> C
    dp = [[(0, 0) for _ in range(3)] for _ in range(n + 1)]
    pegs = ['A', 'B', 'C']
    
    # 初始化 i = 1 的基础情况（只有1个盘子时的移动策略）
    for u in range(3):
        # 另外两根柱子
        dest1 = (u + 1) % 3
        dest2 = (u + 2) % 3
        
        move1 = pegs[u] + pegs[dest1]
        move2 = pegs[u] + pegs[dest2]
        
        # 比较两个合法移动的优先级，选择优先级高的那个
        if prio_map[move1] < prio_map[move2]:
            dp[1][u] = (1, dest1)
        else:
            dp[1][u] = (1, dest2)
            
    # 动态规划递推 2 到 n 个盘子
    for i in range(2, n + 1):
        for u in range(3):
            # 顶部的 i-1 个盘子从 u 移走到 v1
            s1, v1 = dp[i-1][u]
            
            # 此时剩下的那根空柱子必定是 w
            # 因为 0 + 1 + 2 = 3，所以用 3 - u - v1 就可以得到另外一根柱子
            w = 3 - u - v1  
            
            # 第 i 个盘子必定从 u 移动到 w （花费1步）
            # 接着 i-1 个盘子从 v1 继续移动，会自然地落到 v2 上
            s2, v2 = dp[i-1][v1]
            
            if v2 == w:
                # 刚好落在了第 i 个盘子上，合并完成
                dp[i][u] = (s1 + 1 + s2, w)
            else:
                # 没有落在第 i 个盘子上（回到了 u），需要把第 i 个盘子移到 v1，
                # 然后 i-1 个盘子再从 u 移到 v1
                dp[i][u] = (2 * s1 + s2 + 2, v1)
                
    # 输出将 n 个盘子从 0 号柱（A柱）移出的总步数
    print(dp[n][0][0])

if __name__ == '__main__':
    Hanoi()

## H 马步距离

In [ ]:
## add your code here
import sys

def get_knight_distance(x_start, y_start, x_end, y_end):
    # 计算两点在 x 轴和 y 轴上的绝对差
    dx = abs(x_start - x_end)
    dy = abs(y_start - y_end)
    
    # 确保 dx >= dy，利用棋盘对称性
    if dx < dy:
        dx, dy = dy, dx
        
    # 处理两个特殊角落情况
    if dx == 1 and dy == 0:
        return 3
    if dx == 2 and dy == 2:
        return 4
        
    # 核心 O(1) 数学公式
    # (dx+1)//2 相当于 ceil(dx/2)
    # (dx+dy+2)//3 相当于 ceil((dx+dy)/3)
    steps = max((dx + 1) // 2, (dx + dy + 2) // 3)
    
    # 奇偶性修正：每步都会改变 (dx+dy) 的奇偶性
    if steps % 2 != (dx + dy) % 2:
        steps += 1
        
    return steps

def compute():
    # 读取所有输入
    data = sys.stdin.read().split()
    if not data:
        return
        
    px = int(data[0])
    py = int(data[1])
    qx = int(data[2])
    qy = int(data[3])
    
    # 输出结果
    print(get_knight_distance(px, py, qx, qy))

if __name__ == '__main__':
    compute()

## I 直方图最大矩形

In [ ]:
## add your code here
class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        # write code here
        extended_heights = heights + [0]
        index_stack = []
        best_area = 0

        for right_bound, current_height in enumerate(extended_heights):
            while index_stack and current_height < extended_heights[index_stack[-1]]:
                top_index = index_stack.pop()
                rectangle_height = extended_heights[top_index]

                if index_stack:
                    left_bound = index_stack[-1]
                    rectangle_width = right_bound - left_bound - 1
                else:
                    rectangle_width = right_bound

                area = rectangle_height * rectangle_width
                best_area = max(best_area, area)

            index_stack.append(right_bound)

        return best_area

## J 消防局的设立

In [ ]:
## add your code here
import sys

def build_fireStation():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    n = int(input_data[0])
    
    # 特判只有一个基地的情况
    if n == 1:
        print(1)
        return

    # 构建无向图的邻接表
    adj = [[] for _ in range(n + 1)]
    for i in range(2, n + 1):
        u = i
        v = int(input_data[i - 1])
        adj[u].append(v)
        adj[v].append(u)

    # 利用 BFS 确定父子关系，并获取自顶向下的节点遍历顺序
    order = []
    visited = [False] * (n + 1)
    queue = [1]
    visited[1] = True
    children = [[] for _ in range(n + 1)]

    head = 0
    while head < len(queue):
        u = queue[head]
        head += 1
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                visited[v] = True
                children[u].append(v)
                queue.append(v)

    # 翻转顺序，得到自底向上（后序）的遍历顺序
    order.reverse()

    INF = int(1e9)
    NINF = int(-1e9)

    f = [INF] * (n + 1)
    g = [0] * (n + 1)
    ans = 0

    # 树形 DP 自底向上递推
    for u in order:
        f_u = INF
        g_u = 0

        # 从子节点转移状态
        for v in children[u]:
            if f[v] + 1 < f_u:
                f_u = f[v] + 1
            if g[v] + 1 > g_u:
                g_u = g[v] + 1

        # 如果子树中的消防局足以覆盖子树中最远的未覆盖节点
        if f_u + g_u <= 2:
            g_u = NINF

        # 如果最远的未覆盖节点距离已经达到 2，必须在当前节点建站
        if g_u == 2:
            ans += 1
            f_u = 0
            g_u = NINF

        f[u] = f_u
        g[u] = g_u

    # 处理根节点残留的未覆盖节点
    if g[1] >= 0:
        ans += 1

    # 输出最少需要的消防局数量
    print(ans)

if __name__ == '__main__':
    build_fireStation()